# Sampling-density stress test: Hellinger ETW versus standard DTW

We fix a continuous latent signal \(x\) and a smooth true warp \(\alpha\),

\[
f(s)=x(s),\qquad g(t)=x(\alpha(t)).
\]

The two experiments differ **only** in their sampling densities.  In experiment A, \(f\) is sampled densely late and \(g\) densely early; in experiment B, the densities are swapped.

Both algorithms use the same RBF pairwise geometry. Hellinger ETW also uses the physical interval durations. Standard DTW applies the usual three-neighbour recurrence to the sampled index sequences and ignores those durations.

**Scope.** This is a deliberately visible stress test, not a theorem that the restricted block ETW recurrence is invariant to every resampling scheme. The purpose is to isolate one regime in which duration weighting materially improves stability.

In [ ]:
# Main parameters.
N = 80                 # number of intervals in each sampled series
BETA = 2.0             # contrast in d_beta(t) proportional to exp(beta*(2t-1))
SIGMA = 0.25           # RBF similarity scale
RUN_SWEEP = True       # optional density-contrast sweep

# The chosen continuous signal has three asymmetric landmarks.  This avoids a
# completely ambiguous periodic example while still leaving room for a
# sampling-count distortion in standard DTW.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / "hellinger_etw.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "hellinger_etw.py").exists():
    raise RuntimeError("Start Jupyter from the repository root or examples/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.sampling_rate_comparison import (
    density_contrast_sweep,
    run_swapped_density_comparison,
    write_result_bundle,
)

comparison = run_swapped_density_comparison(
    n=N,
    beta=BETA,
    sigma=SIGMA,
    use_numba=True,
)
print("Run complete.")

In [ ]:
# Numerical summary.
for result in comparison.experiments:
    print(result.label)
    print(f"  Hellinger warp RMSE to truth: {result.hellinger_rmse:.4f}")
    print(f"  standard DTW RMSE to truth:   {result.dtw_rmse:.4f}")
print()
print(f"Between-experiment Hellinger RMSE: {comparison.hellinger_between_rmse:.4f}")
print(f"Between-experiment DTW RMSE:       {comparison.dtw_between_rmse:.4f}")

## Main comparison

The left panels show the same continuous \(f\) and \(g\) with different sample locations. The right panels show the recovered physical-time map \(t\mapsto \widehat\alpha(t)\).

For DTW, the displayed map assigns each \(g\)-sample the mean physical time of the \(f\)-samples matched to it. This is only a visualization of the standard DTW path; it does not change the DTW optimization.

In [ ]:
output_dir = ROOT / "user_runs" / "sampling_rate_comparison"

if RUN_SWEEP:
    sweep_rows = density_contrast_sweep(
        n=N,
        beta_values=[0.0, 0.4, 0.8, 1.2, 1.6, 1.8, 2.0, 2.2, 2.4],
        sigma=SIGMA,
        use_numba=True,
    )
else:
    sweep_rows = [
        {
            "beta": BETA,
            "hellinger_worst_rmse": max(x.hellinger_rmse for x in comparison.experiments),
            "dtw_worst_rmse": max(x.dtw_rmse for x in comparison.experiments),
            "hellinger_between_rmse": comparison.hellinger_between_rmse,
            "dtw_between_rmse": comparison.dtw_between_rmse,
        }
    ]

files = write_result_bundle(
    comparison,
    sweep_rows,
    output_dir=output_dir,
    prefix=f"sampling_rate_comparison_N{N}_beta{str(BETA).replace('.', 'p')}",
)

display(Image(filename=str(files["main_figure"])))

## Stability summary

The first two panels overlay the two recovered warps. The final panel varies the density contrast \(\beta\). It is included to show that the selected example is not merely a single pair of sample grids; the sharp DTW failure appears once the opposing densities become sufficiently concentrated.

In [ ]:
display(Image(filename=str(files["stability_figure"])))

In [ ]:
print("Saved files:")
for label, path in files.items():
    print(f"  {label:18s} {path.relative_to(ROOT)}")
print("\nShareable bundle:")
print(files["return_zip"])

## Chosen construction

- **Latent signal:** three unequal Gaussian-shaped landmarks plus a small linear drift.
- **True warp:** the normalized integral of a smooth positive speed, so \(\alpha\) is nontrivial and strictly increasing.
- **Sampling density:** \(d_\beta(t)\propto e^{\beta(2t-1)}\). The two experiments swap \(+\beta\) and \(-\beta\) between \(f\) and \(g\).
- **Pairwise comparison:** \(C(x,y)=\exp(-(x-y)^2/(2\sigma^2))\). Hellinger ETW maximizes this duration-weighted similarity; standard DTW minimizes \(1-C\) on the index grid.

A timestamp-aware or continuous-DTW baseline would be a useful stronger comparison in a later empirical section. The present notebook isolates ordinary index-based DTW.